In [ ]:
pip install streamlit pandas langchain-experimental langchain-openai

In [ ]:
pip install --upgrade langchain-community

In [ ]:
pip install langchain-experimental langchain-community langchain-openai pandas streamlit

In [ ]:
pip install langchain-ollama

In [ ]:
pip install tabulate

In [ ]:
!pip install matplotlib seaborn openpyxl

In [1]:
%%writefile app.py
import warnings
warnings.filterwarnings("ignore")

import streamlit as st
import pandas as pd
from langchain_ollama import ChatOllama

st.set_page_config(page_title="Talk2Data", layout="wide")
st.title("📊 AI Data Analyst Tool ")

@st.cache_resource
def load_llm():
    return ChatOllama(model="llama3.2", temperature=0)

llm = load_llm()

uploaded_file = st.sidebar.file_uploader("Upload your CSV file", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    
    # Pre-calculate Revenue column to make Llama's job 100% accurate & easy
    if 'Quantity' in df.columns and 'UnitPrice' in df.columns:
        df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']
        
    st.subheader("Data Preview")
    st.dataframe(df.head())
    
    user_question = st.text_input("Apna data question poochhein:")
    
    if user_question:
        with st.spinner("Analyzing data with Llama..."):
            try:
                prompt = f"""
You are a Python Pandas Data Analyst.
You are given a DataFrame named `df` with columns: {list(df.columns)}

User Question: {user_question}

Write ONLY ONE line of valid, executable Python Pandas code to answer the question.
STRICT RULES:
1. Do NOT write any English text or explanations.
2. Use valid Pandas methods like .groupby(), .sum(), .sort_values(), .head(), .value_counts().
3. Do NOT invent invalid syntax.

Example 1: df.groupby('Country')['TotalRevenue'].sum().sort_values(ascending=False).head(5)
Example 2: df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5)
"""
                response = llm.invoke(prompt)
                code_to_exec = response.content.strip().replace("```python", "").replace("```", "").strip()
                
                # Take only the first non-empty line of code
                code_lines = [line.strip() for line in code_to_exec.split('\n') if line.strip()]
                clean_code = code_lines[0] if code_lines else code_to_exec
                
                st.subheader("Generated Pandas Code:")
                st.code(clean_code, language="python")
                
                # Execute Python Code directly on Pandas
                result = eval(clean_code)
                
                st.subheader("Answer:")
                st.write(result)
                
            except Exception as e:
                st.error(f"Execution Error: {e}")
else:
    st.info("Kripya sidebar se apni CSV file upload karein.")

Overwriting app.py


In [ ]:
!streamlit run app.py